# Google - Gmail Label Usage and User Efficiency

In [1]:
import pandas as pd  
import numpy as np
import polars as pl
from datetime import date

In [3]:
df_emails = pd.read_csv('../Data/003/emails.csv', )
df_email_labels = pd.read_csv('../Data/003/email_labels.csv', parse_dates=['created_date'])

pl_emails = pl.read_csv('../Data/003/emails.csv', )
pl_email_labels = pl.read_csv('../Data/003/email_labels.csv', try_parse_dates = True)

# Pregunta 1

### ¿Puedes averiguar la cantidad de etiquetas creadas por cada usuario? Estamos interesados en entender cuántas etiquetas crean los usuarios normalmente para gestionar sus correos electrónicos

```SQL
SELECT
    user_id,
    COUNT(*)
FROM email_labels
GROUP BY user_id;
```

In [10]:
res = df_email_labels.groupby('user_id')['label_id'].count().reset_index(name='total_etiquetas')

In [15]:
res = pl_email_labels.group_by(pl.col('user_id')).agg(
    pl.len().alias('total_etiquetas')
).sort("total_etiquetas", descending=True)

# Pregunta 2

### Tu equipo quiere saber qué etiquetas tienen más de 5 correos electrónicos asignados. ¿Puedes obtenerlas?

```SQL
SELECT
    label_id,
    COUNT(email_id) Total_Correo_Asignados
FROM emails
GROUP BY label_id
HAVING COUNT(email_id) > 5
```

In [20]:
res = df_emails.groupby('label_id').agg(
    total_correos=('email_id','count')
).reset_index()

res = res[res['total_correos'] > 5]

res


,label_id,total_correos
3,4,10
14,15,6


In [21]:
res = pl_emails.group_by('label_id').agg(
    pl.col('email_id').count().alias('total_correos')
).filter(
    pl.col('total_correos') > 5
)

res
res

label_id,total_correos
i64,u32
15,6
4,10


# Pregunta 3

### Para las etiquetas creadas en octubre de 2024, determina la cantidad de correos electrónicos asociados a cada una. Si alguna etiqueta creada en octubre no tiene correos asociados, inclúyela de todos modos en el resultado. Esto nos ayudará a entender la distribución del uso de correos en las etiquetas.

```SQL
SELECT
    l.label_id,
    COUNT(e.email_id)
FROM email_labels AS l
LEFT JOIN emails AS e ON l.label_id =e.label_id
WHERE l.created_date BETWEEN '2024-10-01' AND '2024-10-31'
GROUP BY l.label_id
```

In [24]:
df_merge = df_email_labels.merge(
    df_emails,
    on='label_id',
    how='left'
)

df_oct = df_merge[
    df_merge['created_date'].between('2024-10-01','2024-10-31')
].reset_index()

res = df_oct.groupby('label_id')['email_id'].count().reset_index(name='total_correos')

In [27]:
res = pl_email_labels.join(
    pl_emails,
    on = 'label_id',
    how = 'left'
).filter(
    pl.col('created_date').is_between(date(2024,10,1),date(2024,10,31))
)

res = res.group_by('label_id').agg(
    pl.col('email_id').count().alias('total_correos')
)

res

label_id,total_correos
i64,u32
1,2
6,3
3,5
2,3
4,10
5,3
